# 04b — Training Loop (smoke test)

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 4b — Training loop con AMP + cosine schedule + logging local

## Objetivo

Validar que el training loop está bien armado **antes** del run de pre-training real. Específicamente:

1. Cargar los pickles del corpus generados en Fase 4a.
2. Instanciar el modelo con `max_seq_length=80` (justificación: p99 de Fase 4a fue 64) → batch grande, throughput alto.
3. Correr **200 pasos** de smoke test con MLM puro sobre un subset del pretrain.
4. Verificar que la loss **baja** (de ~ln(4521) ≈ 8.4 hacia ~5), la perplexity también, y la MLM accuracy sube.
5. Verificar que el checkpoint guardado se puede recargar exactamente.
6. Correr la suite de tests de training (5 + 5 + 5 = 15 tests nuevos).

Después del smoke test, si todo se ve bien, en Fase 4c vamos a correr el **pre-training real**: ~1 época sobre los 11.852 docs (≈ 370 pasos a batch=32, o ~93 a batch=128).

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CKPT_DIR.mkdir(exist_ok=True)

assert (CORPUS_DIR / 'pretrain.pkl').exists(), 'Faltan los pickles de Fase 4a'
print(f'✓ Corpus encontrado en {CORPUS_DIR}')

In [ ]:
from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer
from data.dataset import MatchDataset
from data.collator import MLMCollator
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

print('torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

---
## 2. Cargar vocab + corpus

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)

# max_seq_length=80: p99 de Fase 4a fue 64 → margen de 16 tokens.
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

with open(CORPUS_DIR / 'pretrain.pkl', 'rb') as f:
    pretrain_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)

print(f'pretrain docs: {len(pretrain_docs):,}')
print(f'val docs:      {len(val_docs):,}')

---
## 3. Dataset + DataLoader

Para el smoke usamos batch_size=64 y un subset chico (1000 docs). Para Fase 4c vamos a usar el corpus completo.

In [ ]:
import random
random.seed(0)
smoke_docs = random.sample(pretrain_docs, 1000)

ds_train = MatchDataset(smoke_docs, tokenizer)
ds_val   = MatchDataset(val_docs[:200], tokenizer)

collator = MLMCollator(vocab, mlm_probability=0.15, seed=42)

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=0)

print(f'Train batches: {len(train_loader)}  (batch_size={BATCH_SIZE})')
print(f'Val batches:   {len(val_loader)}')

# Sanity: peek at one batch shape
sample_batch = next(iter(train_loader))
print(f'\nBatch token_ids shape: {tuple(sample_batch.token_ids.shape)}')
print(f'Batch attention_mask sum/total: {sample_batch.attention_mask.sum().item()} / {sample_batch.attention_mask.numel()}')

---
## 4. Instanciar modelo + Trainer

In [ ]:
model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80,   # ← actualizado para coincidir con el tokenizer
    num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)
print(f'Modelo: {model.num_parameters():,} parámetros')

trainer_config = TrainerConfig(
    lr=5e-4, weight_decay=0.01,
    grad_clip_norm=1.0, warmup_ratio=0.1,
    max_steps=200,            # ← smoke test
    mixed_precision=True,     # AMP en GPU
    log_every=10,
    eval_every=100,           # 1 eval a mitad y otra al final
    save_every=0,             # no guardamos intermedios en smoke
    save_best=True,
    output_dir=str(CKPT_DIR),
    run_name='smoke_test_pretrain',
    seed=42,
)
loss_spec = LossSpec(use_mlm=True, use_result=False, use_score=False)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=trainer_config,
    loss_spec=loss_spec,
)
print(f'Device: {trainer.device}')
print(f'AMP enabled: {trainer.use_amp}')
print(f'Output dir: {trainer.output_dir}')

---
## 5. Smoke run (200 pasos)

**Loss inicial esperada:** ~ln(vocab_size) = ln(4521) ≈ 8.42 — el modelo aleatorio está en máxima entropía sobre el vocabulario.

**Después de 200 pasos esperamos:** loss en ~5-6, perplexity ~150-400, MLM accuracy ~15-25%.

Si la loss no baja de 8.0 en 200 pasos, hay un bug (sospechar: lr, gradient flow, masking).

In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Smoke test completado en {elapsed:.1f}s ({200/elapsed:.1f} step/s)')

---
## 6. Curvas de loss / accuracy / lr

In [ ]:
# Leer el log JSONL
log_path = trainer.log_path
rows = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
train_rows = [r for r in rows if 'phase' not in r]
eval_rows  = [r for r in rows if r.get('phase') == 'eval']

steps_train = [r['step'] for r in train_rows]
loss_train  = [r['loss'] for r in train_rows]
ppl_train   = [r['mlm_perplexity'] for r in train_rows]
acc_train   = [r['mlm_acc'] for r in train_rows]
lr_train    = [r['lr'] for r in train_rows]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(steps_train, loss_train, label='train')
if eval_rows:
    axes[0, 0].plot([r['step'] for r in eval_rows], [r['val_loss'] for r in eval_rows], 'o-', label='val')
axes[0, 0].axhline(np.log(len(vocab)), linestyle='--', color='red', alpha=0.5,
                   label=f'random baseline = ln({len(vocab)}) = {np.log(len(vocab)):.2f}')
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('loss'); axes[0, 0].set_title('MLM loss')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(steps_train, ppl_train, label='train')
if eval_rows:
    axes[0, 1].plot([r['step'] for r in eval_rows], [r['val_mlm_perplexity'] for r in eval_rows], 'o-', label='val')
axes[0, 1].set_xlabel('step'); axes[0, 1].set_ylabel('perplexity')
axes[0, 1].set_title('Perplexity'); axes[0, 1].set_yscale('log')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(steps_train, acc_train, label='train')
if eval_rows:
    axes[1, 0].plot([r['step'] for r in eval_rows], [r['val_mlm_acc'] for r in eval_rows], 'o-', label='val')
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylabel('top-1 accuracy')
axes[1, 0].set_title('MLM accuracy'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(steps_train, lr_train, color='purple')
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylabel('lr')
axes[1, 1].set_title('Learning rate schedule')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico cuantitativo: ¿bajó la loss?
print('=== Diagnóstico de aprendizaje ===')
print(f'Random baseline (ln(V)):      {np.log(len(vocab)):.4f}')
print(f'Loss inicial (paso {steps_train[0]:>3}):     {loss_train[0]:.4f}  →  ppl {ppl_train[0]:8.2f}  acc {acc_train[0]:.4f}')
print(f'Loss final   (paso {steps_train[-1]:>3}):     {loss_train[-1]:.4f}  →  ppl {ppl_train[-1]:8.2f}  acc {acc_train[-1]:.4f}')
print(f'Reducción de loss:             {100 * (loss_train[0] - loss_train[-1]) / loss_train[0]:.1f}%')
if eval_rows:
    print(f'\nVal final: loss={eval_rows[-1]["val_loss"]:.4f}  ppl={eval_rows[-1]["val_mlm_perplexity"]:.2f}  acc={eval_rows[-1]["val_mlm_acc"]:.4f}')

ok_signal = loss_train[-1] < loss_train[0] - 0.5
print(f'\n{"✓" if ok_signal else "⚠"} Señal de aprendizaje: {"OK" if ok_signal else "WEAK"} (esperábamos al menos −0.5 de loss)')

---
## 7. Checkpoint roundtrip

Verificamos que podemos guardar y recargar el modelo exactamente.

In [ ]:
from copy import deepcopy
ckpt_path = trainer.output_dir / 'final.pt'
print(f'Checkpoint guardado: {ckpt_path}  ({ckpt_path.stat().st_size / 1024 / 1024:.1f} MB)')

# Construir un trainer nuevo + recargar
model2 = D10Sformer(model_config)
trainer2 = Trainer(
    model=model2, train_loader=train_loader, val_loader=val_loader,
    config=trainer_config, loss_spec=loss_spec,
)
trainer2.load_checkpoint(ckpt_path)

# Comparar todos los parámetros
match = True
for (n1, p1), (n2, p2) in zip(trainer.model.named_parameters(), trainer2.model.named_parameters()):
    if not torch.allclose(p1, p2, atol=1e-6):
        match = False
        print(f'✗ Mismatch en {n1}')
        break
print(f'{"✓" if match else "✗"} Roundtrip de parámetros: {"idénticos" if match else "DIFERENTES"}')
print(f'  step restaurado: {trainer2.step}  (debe ser {trainer.step})')
print(f'  best_val_loss:   {trainer2.best_val_loss}')

---
## 8. Suite de tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest',
     str(ROOT / 'tests' / 'test_scheduler.py'),
     str(ROOT / 'tests' / 'test_training_metrics.py'),
     str(ROOT / 'tests' / 'test_trainer.py'),
     '-v', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout[-4000:])
if result.stderr:
    print('STDERR:', result.stderr[-800:])
print(f'Exit code: {result.returncode}')

---
## 9. Conclusiones de Fase 4b (llenar al final)

- [ ] Loss inicial (paso 10): _____
- [ ] Loss final (paso 200): _____
- [ ] Reducción de loss: _____% (esperado ≥ 30%)
- [ ] Perplexity inicial / final: _____ / _____
- [ ] MLM accuracy final: _____ (esperado > 0.10)
- [ ] Val loss final: _____
- [ ] Steps por segundo: _____ (T4 esperado: 5-15 step/s con batch 64)
- [ ] Tests pasados / total: _____ / 15
- [ ] Checkpoint roundtrip OK: _____

**Si todo verde → Fase 4c (training real):** 3-5 épocas sobre el corpus completo, lo cual son ~550 a ~900 pasos a batch=64. Estimación T4: 60-90 minutos.